# Original checkpoint: test-time batch-normalization recalibration

**No retraining.** This uses your original **4,480,474-parameter** final model and the
2,000 unlabeled test images. The user confirmed organizer permission for test-time BN
adaptation. Do not use this workflow in competitions where such adaptation is prohibited.

Select a GPU runtime. Upload **data.zip** and the original **final.pt** into `/content`
using the Files panel. Use `final.pt` from your original `runs/final`, **not** the AugMix
checkpoint from HPC. Alternatively, edit the paths below to point to mounted Drive files.

The procedure replaces BN running means/variances using one pass over fixed shuffled
batches of 100 test images (plus their flips), then freezes BN again for prediction.
It does not use test labels, optimizer steps, or gradients. Convolution, classifier,
and BN scale/bias parameters stay exactly unchanged. This is transductive adaptation:
all supplied test images contribute to the statistics. It may improve or hurt accuracy.

Outputs go into a new timestamped folder; your original checkpoint/submission are preserved.

In [ ]:
from pathlib import Path
import json, subprocess, sys, time
import torch

DATA_ZIP = Path('/content/data.zip')
CHECKPOINT = Path('/content/final.pt')
ORGANIZERS_ALLOW_TEST_BN_ADAPTATION = True  # User confirmed this permission.
assert ORGANIZERS_ALLOW_TEST_BN_ADAPTATION
assert DATA_ZIP.is_file(), 'Upload data.zip to /content or edit DATA_ZIP.'
assert CHECKPOINT.is_file(), 'Upload the original final.pt to /content or edit CHECKPOINT.'
assert torch.cuda.is_available(), 'Choose a GPU runtime.'
checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
assert 'state_dict' in checkpoint, 'Use the exported final.pt, not last.pt.'
assert checkpoint['parameter_count'] == 4480474, 'This notebook expects the original 4,480,474-parameter model.'
assert checkpoint['full_data'] and checkpoint['epoch'] == checkpoint['config']['epochs'], 'Use the completed original full-data checkpoint.'
assert not checkpoint.get('bn_adaptation'), 'Start with your original unadapted checkpoint.'
print('Original checkpoint:', {k:checkpoint[k] for k in ['epoch','weights','parameter_count']})
print('GPU:', torch.cuda.get_device_name(0))
del checkpoint

## Load code and prepare images

Preparation checks the supplied ZIP's format and creates caches. The adaptation itself
reads only `test_images.npy`, with no labels; it does not retrain on the training images.

In [ ]:
WORK = Path('/content') / ('bn_adaptation_' + time.strftime('%Y%m%d_%H%M%S'))
REPO = WORK / 'code'
WORK.mkdir(parents=True, exist_ok=False)
subprocess.run(['git','clone','https://github.com/mj3664-web/cifar-resnet-colab.git',str(REPO)],check=True)
def run(module, *args):
    subprocess.run([sys.executable,'-u','-m',module,*map(str,args)],cwd=REPO,check=True)
CACHE = WORK / 'cache'
run('src.data','--zip',DATA_ZIP,'--out',CACHE)
OUTPUT = WORK / 'results'
OUTPUT.mkdir()
commit = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
(OUTPUT/'code_commit.txt').write_text(commit)

## Recalibrate BN, freeze it, and generate predictions

This uses a fixed recipe, not a sweep selected against hidden labels. BatchNorm layers
alone enter training mode during calibration; dropout remains disabled. Calibration is
float32. All 20 calibration batches have equal size, each with 100 originals + 100 flips.
Final prediction uses `model.eval()` with fixed adapted buffers and the checkpoint's
horizontal-flip probability averaging. Only one model is used; parameter count is unchanged.

In [ ]:
ADAPTED = OUTPUT / 'original_bn_adapted.pt'
run('src.bn_adapt','--data',CACHE,'--checkpoint',CHECKPOINT,'--out',ADAPTED,
    '--batch-size','100','--seed','2026','--workers','2','--allow-test-bn-adaptation')
run('src.predict','--data',CACHE,'--checkpoint',ADAPTED,
    '--out',OUTPUT/'submission.csv','--workers','2')
print('New submission:', OUTPUT/'submission.csv')
print('Adapted checkpoint:', ADAPTED)

## Download the new submission and reproduction files

Submit the new CSV through the official competition page if desired. The notebook cannot
calculate its Kaggle accuracy. Keep your original submission to compare official results.
The archive includes the adapted checkpoint, metadata, and CSV; save it before the runtime
is deleted. No model optimizer state is needed to reproduce inference from this checkpoint.

In [ ]:
from google.colab import files
import shutil
files.download(str(OUTPUT/'submission.csv'))
archive = shutil.make_archive(str(WORK/'bn_adaptation_results'),'zip',root_dir=OUTPUT)
files.download(archive)